# RAG-Fusion

**RAG-Fusion** is an advanced retrieval technique that improves upon Multi-Query Retrieval by ranking retrieved documents instead of simply merging them. It generates multiple search queries from the original user question, retrieves documents for each query, and combines the results using **Reciprocal Rank Fusion (RRF)** to identify the most relevant context.

---

## Why RAG-Fusion?

Traditional Multi-Query Retrieval retrieves documents for multiple query variations and merges the results. However, all retrieved documents are treated almost equally.

RAG-Fusion addresses this limitation by assigning higher importance to documents that consistently appear at the top of retrieval results across multiple queries.

---

## Workflow

```text
                User Question
                      │
                      ▼
         Generate Multiple Queries
                      │
      ┌────────┬────────┬────────┐
      ▼        ▼        ▼        ▼
   Query1   Query2   Query3   Query4
      │        │        │        │
      ▼        ▼        ▼        ▼
 Retrieve  Retrieve  Retrieve  Retrieve
      │        │        │        │
      └────────┴────────┴────────┘
                 │
                 ▼
     Reciprocal Rank Fusion (RRF)
                 │
                 ▼
      Top Ranked Documents
                 │
                 ▼
                LLM
                 │
                 ▼
           Final Answer
```

---

## Advantages

- Improves retrieval accuracy by ranking retrieved documents.
- Reduces noisy or less relevant retrieval results.
- Prioritizes documents that consistently appear across multiple searches.
- Produces more reliable context for answer generation.
- Often outperforms simple Multi-Query Retrieval.

---

## Key Difference from Multi-Query Retrieval

| Multi-Query Retrieval               | RAG-Fusion                                                |
| ----------------------------------- | --------------------------------------------------------- |
| Generates multiple query variations | Generates multiple query variations                       |
| Retrieves documents for each query  | Retrieves documents for each query                        |
| Merges retrieved documents          | Ranks retrieved documents using RRF                       |
| Removes duplicate documents         | Prioritizes the most relevant documents before generation |

---

## One-Line Definition

> **RAG-Fusion is a retrieval technique that generates multiple search queries and combines their retrieved results using Reciprocal Rank Fusion (RRF) to produce better-ranked context for the LLM.**


In [9]:
# 01_basic_rag.ipynb run gareko
%run ./pipeline/01_basic_rag.ipynb

# 02_indexing_rag.ipynb run gareko
%run ./pipeline/02_indexing_rag.ipynb

1.3.13


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 8d42537b-1a3e-474d-b1cb-69d180e225f8)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558


In [10]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion ko lagi prompt template create gareko
template = """
You are a helpful assistant that generates multiple search queries based on a single input query.

Generate multiple search queries related to the following question:

{question}

Output (4 queries):
"""

# RAG-Fusion ko prompt template create gareko
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# Local Ollama LLM use gareko (OpenAI ko satta)
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Prompt → LLM → Output Parser → Multiple queries generate garne chain
generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    # Generate bhayeka queries lai newline ko basis ma list ma split gareko
    | (lambda x: x.split("\n"))
)

In [12]:
from langchain_core.load import dumps, loads


# Reciprocal Rank Fusion (RRF) algorithm implement gareko
def reciprocal_rank_fusion(results: list[list], k=60):
    """
    Multiple retrieved document lists lai RRF use garera rerank garne function
    """

    # Pratyek unique document ko fused score store garna dictionary create gareko
    fused_scores = {}

    # Pratyek query bata retrieve bhayeko documents iterate garne
    for docs in results:

        # Document ra tesko rank (position) line
        for rank, doc in enumerate(docs):

            # Document lai string ma convert gareko
            doc_str = dumps(doc)

            # Document pahila dictionary ma chaina bhane initialize gareko
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            # RRF formula use garera document ko score update gareko
            fused_scores[doc_str] += 1 / (rank + k)

    # Highest score bhayeka documents lai descending order ma sort gareko
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True,
        )
    ]

    # Reranked documents return gareko
    return reranked_results


# Multi-query generate garne → Retrieval garne → RRF use garera rerank garne chain
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion


# RAG-Fusion retrieval execute gareko
docs = retrieval_chain_rag_fusion.invoke({"question": question})


# Retrieve bhayeka reranked documents ko number print garne
print(len(docs))

3


C:\Users\Acer\AppData\Local\Temp\ipykernel_21776\2268467975.py:31: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)
C:\Users\Acer\AppData\Local\Temp\ipykernel_21776\2268467975.py:31: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [13]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# RAG ko prompt template create gareko
template = """
Answer the following question based only on the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


# Final RAG-Fusion chain create gareko
final_rag_chain = (
    {
        # RAG-Fusion bata rerank bhayeko documents context ma pathaune
        "context": retrieval_chain_rag_fusion,
        # User ko original question prompt ma pathaune
        "question": itemgetter("question"),
    }
    # Context ra question prompt template ma inject garne
    | prompt
    # Local Ollama LLM bata final answer generate garne
    | llm
    # LLM ko output lai plain text ma convert garne
    | StrOutputParser()
)


# User ko question RAG-Fusion pipeline ma pathaune
response = final_rag_chain.invoke({"question": question})

# Final generated answer print garne
print(response)

C:\Users\Acer\AppData\Local\Temp\ipykernel_21776\2268467975.py:31: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


There is no information provided about your preferences for pets or any other topic. The context only discusses AI agents, models, and algorithms, with no mention of personal preferences or interests. Therefore, it's not possible to answer the question based on the provided context.
